# Natural Language Processing (72 pts)



* List of modules to load: miniconda academic-ml/fall-2025

* Pre-Launch Command: conda activate fall-2025-pyt

* When you would load 'conll2003train.txt', instead load '/projectnb/ds340/materials/conll2003train.txt'.

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.2 MB/s eta 0:00:00


# Part I.  Named Entity Recognition (25 pts)

Named entity recognition (NER) is a classic NLP task where proper nouns and their types must be extracted from text.  The CONLL 2003 dataset labels entities in text as PER (person), ORG (organization), LOC (location), MISC (proper noun but none of the above), or O (nothing).  A classifier trained on this data can label each word in a sentence as belonging to one of these categories.

In this section, we'll use word2vec vectors to classify each word.  Word2vec doesn't use any context from the rest of the sentence, but the task of identifying proper nouns as places or people may not need a lot of context.

In [2]:
# CONLL (Computational Natural Language Learning) 2003
# data from:
# https://data.deepai.org/conll2003.zip
# description of data:
# https://huggingface.co/datasets/eriktks/conll2003

from google.colab import files
# Pick conll2003train.txt for full training
uploaded = files.upload()


Saving conll2003train.txt to conll2003train.txt


In [3]:
!ls

conll2003train.txt  sample_data


In [ ]:
import csv

def load_ner_data(filename):
  lines = []
  with open(filename, mode='r') as myfile:
    spacereader = csv.reader(myfile, delimiter=' ')
    working_sentence = []
    working_ner_tags = []
    for row in spacereader:
      if len(row) == 0:
        if len(working_sentence) > 0:
          lines.append((working_sentence, working_ner_tags))
          working_sentence = []
          working_ner_tags = []
      elif len(row) == 4:
        if row[0] != '-DOCSTART-':
          working_sentence.append(row[0])
          working_ner_tags.append(process_ner_tag(row[3]))
  return lines

def process_ner_tag(tag):
  if tag == 'O': 
    return 0
  tag = tag[2:] 
  tag_dict = {
      'PER': 1,
      'ORG': 2,
      'LOC': 3,
      'MISC': 4
  }
  return tag_dict[tag]

The following line should load the relevant data as a list of tuples, where the first element of each tuple is a list of words in a sentence, and the second element of each tuple is a list of the words' proper numerical labels.

In [5]:
all_tuples = load_ner_data('conll2003train.txt')

In [6]:
len(all_tuples)

14038

In [7]:
print(all_tuples[0])

(['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], [2, 0, 4, 0, 0, 0, 4, 0, 0])


For faster processing, we'll just work with the first 1000 sentences.

In [8]:
MAX_SENTENCES = 1000
tuples = all_tuples[:MAX_SENTENCES]

Write a function words_to_word2vec() that converts the conll2003 data into a feature matrix and label array of the kind expected by scikit-learn.

The first argument to the function should be a list of tuples of the kind produced by load_ner_data().

The second argument to the function should be a model of the kind returned by gensim.downloader.load().  A call creating one of these objects has been provided.

The first return value should be a $W \times 300$ feature matrix, where $W$ is the number of words in all the input sentences combined, and 300 is the number of elements in each vector returned by the word vector model.  If a word can't be found in the word model, the corresponding line should be all zeros.

The second return value should be a 1d $W$-element array of the labels of the words.

In [ ]:
# Approach #1:  Convert every vector using word2vec,
# and train a scikit-learn classifier on these vectors.

import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
import numpy as np
def words_to_word2vec_matrix(tuple_list, wv):
    feature_rows = []
    label_list = []

    for words, word_labels in tuple_list:
        for word, label in zip(words, word_labels):
            if word in wv:
                vec = wv[word]
            else:
                vec = np.zeros(300)
            feature_rows.append(vec)
            label_list.append(label)

    features = np.array(feature_rows)
    labels = np.array(label_list)
    return features, labels

In [ ]:
features, labels = words_to_word2vec_matrix([(['Sonic', 'is', 'fast'], [1, 0, 0])], wv)
print(features.shape) # expect (3, 300)
print(labels.shape) # expect (3,)

(3, 300)
(3,)


 Now, perform a train/test split on your feature matrix and labels (with test_size = 0.1 and random_state=340) and measure the accuracy of a RandomForestClassifier with 200 estimators (and also random_state=340, other settings default) that uses your word2vec matrix as its features.  You can expect roughly 94% accuracy.

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features, labels = words_to_word2vec_matrix(tuples, wv)

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=340
)

rf = RandomForestClassifier(n_estimators=200, random_state=340)
rf.fit(X_train, y_train)

print(f"Accuracy  {rf.score(X_test, y_test):.4f}")  # expect ~0.94


Accuracy  0.9457


I.3, 4 pts) This is a task where "not a proper noun" is a common category and a pretty good guess, inflating accuracy.  Call sklearn.metrics.precision_recall_fscore_support to see precision, recall, and f-scores for each class.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, fscore, support = precision_recall_fscore_support(
    y_test, rf.predict(X_test)
)

class_names = {0: "O", 1: "PER", 2: "ORG", 3: "LOC", 4: "MISC"}

print(f"{'Class':<8} {'Precision':>10} {'Recall':>10} {'F-score':>10} {'Support':>10}")
print("-" * 52)
for i, (p, r, f, s) in enumerate(zip(precision, recall, fscore, support)):
    print(f"{class_names.get(i, str(i)):<8} {p:>10.4f} {r:>10.4f} {f:>10.4f} {s:>10}")


Class     Precision     Recall    F-score    Support
----------------------------------------------------
O            0.9486     0.9978     0.9726        906
PER          0.9806     0.8347     0.9018        121
ORG          0.8511     0.7018     0.7692         57
LOC          0.9351     0.8090     0.8675         89
MISC         0.8889     0.6400     0.7442         25


I.4, 6 pts)  Identify which class had the lowest *precision* and what it means to have that precision.  Then do the same for *recall*.

**TODO precision**
the class with the lowest percesion is ORG, meaning that for organizations the predictor was only able to correctly classify it about 85% pf this time and the rest were false positives.

**TODO recall**
the class with the lowest recall is MISC, meaning that out of the words that are actually MISC the classfier could only find around 64% and it missed the other 36%


We are now going to walk through an example of how attention could be computed in a sentence.  This omits the multiplication by learned matrices, but captures how the main mechanism of attention alters word embeddings.

  Use the word vector model that you used in the last problem to look up a list of vectors for the following two sentences:

* "Turkey closed its borders today."
* "Turkey is a Thanksgiving tradition."

As before, if a word isn't in the model, use a 300 element zero vector.

In [ ]:
import numpy as np

sentence1 = ['Turkey', 'closed', 'its', 'borders', 'today']
sentence2 = ['Turkey', 'is', 'a', 'Thanksgiving', 'tradition']

def sentence_to_vectors(sentence, wv):
    return [wv[word] if word in wv else np.zeros(300) for word in sentence]

vecs1 = sentence_to_vectors(sentence1, wv)
vecs2 = sentence_to_vectors(sentence2, wv)

II.2, 3 pts) For just the word "Turkey", find the dot product of its vector with each other vector in each sentence.  Report which word has the largest dot product in each sentence (besides the word itself).

In [ ]:
turkey_vec = vecs1[0]

dots1 = []
for i, (word, vec) in enumerate(zip(sentence1, vecs1)):
    if i == 0:
        continue
    dot = np.dot(turkey_vec, vec)
    dots1.append((word, dot))
    print(f"Turkey · {word:15s} = {dot:.4f}")

best_word1 = max(dots1, key=lambda x: x[1])[0]
print(f"\nLargest dot product: '{best_word1}'")

Turkey · closed          = 0.7425
Turkey · its             = 0.6607
Turkey · borders         = 2.2991
Turkey · today           = 0.1390

Largest dot product: 'borders'


**TODO note word with largest dot**

the words with the largest dot product is boarders from Turkey · borders

In [ ]:
dots2 = []
for i, (word, vec) in enumerate(zip(sentence2, vecs2)):
    if i == 0:
        continue
    dot = np.dot(turkey_vec, vec)
    dots2.append((word, dot))
    print(f"Turkey · {word:15s} = {dot:.4f}")

best_word2 = max(dots2, key=lambda x: x[1])[0]
print(f"\nLargest dot product: '{best_word2}'")



Turkey · is              = 0.3096
Turkey · a               = 0.0000
Turkey · Thanksgiving    = 2.1852
Turkey · tradition       = 1.1169

Largest dot product: 'Thanksgiving'


**TODO note word with largest dot**
the words with the largest dot product is thanksgiving from Turkey · Thanksgivign

II.3, 3 pts) Use the softmax formula, $\frac{e^{x_i}}{\sum_i e^{x_i}}$, on each element of each of the dot product lists to create a list of weights that sum to 1 in each case.  Don't include the "Turkey" vector dot product in the calculation.

In [ ]:
def softmax(values):
    v = np.array(values)
    v = v - v.max()
    exp_v = np.exp(v)
    return exp_v / exp_v.sum()

raw_dots1 = [d for (_, d) in dots1]
weights1 = softmax(raw_dots1)

for (word, _), w in zip(dots1, weights1):
    print(f"  {word:15s}: {w:.6f}")
print(f"Sum = {weights1.sum():.6f}")

  closed         : 0.138682
  its            : 0.127787
  borders        : 0.657688
  today          : 0.075843
Sum = 1.000000


In [ ]:
raw_dots2 = [d for (_, d) in dots2]
weights2 = softmax(raw_dots2)

for (word, _), w in zip(dots2, weights2):
    print(f"  {word:15s}: {w:.6f}")
print(f"Sum = {weights2.sum():.6f}")

  is             : 0.095231
  a              : 0.069878
  Thanksgiving   : 0.621390
  tradition      : 0.213501
Sum = 1.000000


II.4, 3 pts) Create a single vector for each sentence that is Turkey's attention vector:  the weighted sum of the four vectors that don't correspond to the word "Turkey", where the weights were created by the softmax.

In [ ]:
non_turkey_vecs1 = [vecs1[i] for i in range(len(sentence1)) if i != 0]

attention_vec1 = sum(w * vec for w, vec in zip(weights1, non_turkey_vecs1))
print("Sentence 1 attention vector (first 5 dims):", attention_vec1[:5])

Sentence 1 attention vector (first 5 dims): [-0.09401011  0.08368635  0.19789726  0.12897971 -0.12705293]


In [ ]:
non_turkey_vecs2 = [vecs2[i] for i in range(len(sentence2)) if i != 0]

attention_vec2 = sum(w * vec for w, vec in zip(weights2, non_turkey_vecs2))
print("Sentence 2 attention vector (first 5 dims):", attention_vec2[:5])

Sentence 2 attention vector (first 5 dims): [ 0.10023212  0.11246022 -0.09518583  0.15081948 -0.10344512]


II.5, 4 pts) Run the classifier for part 1 on 3 vectors:  the plain "Turkey" vector, the "Turkey" vector with WEIGHT times the sentence 1 attention vector added, and the "Turkey" vector with WEIGHT times sentence 2 attention vector added.  Experiment with values of WEIGHT until you find a setting where the first sentence's attention-modified Turkey vector is a location, but the second is not.  (The learned value matrices in actual attention can accomplish what WEIGHT is doing here, and more.)

In [ ]:
class_names_full = {0: "O", 1: "PER", 2: "ORG", 3: "LOC", 4: "MISC"}
turkey_vec = vecs1[0]

for weight in np.arange(0.5, 100.0, 0.5):
    pred1 = rf.predict((turkey_vec + weight * attention_vec1).reshape(1, -1))[0]
    pred2 = rf.predict((turkey_vec + weight * attention_vec2).reshape(1, -1))[0]

    if pred1 == 3 and pred2 != 3:
        print(f"WEIGHT = {weight}")
        print(f"  Sentence 1 Turkey → {class_names_full[pred1]}")
        print(f"  Sentence 2 Turkey → {class_names_full[pred2]}")
        break

WEIGHT = 2.5
  Sentence 1 Turkey → LOC
  Sentence 2 Turkey → O


# Part III.  Using a pretrained language model "off-the-shelf" (24 points)

Now, we'll try using a model that is a step up from word vectors - a BERT model that has been trained to produce a vector for each word in the sentence that is informed by attention.  We'll also change tasks, since the Google-News-trained word2vec seemed pretty good already for the CONLL2003 NER task.

The JNLPBA dataset is like CONLL 2003, but labels words as to whether they are words for DNA, RNA, proteins, cell lines, or cell types.

In [22]:
"""
Dataset description: https://huggingface.co/datasets/jnlpba/jnlpba

NLP for labeling biological terms.  Original labels (thanks to
https://medium.com/@raj.pulapakura/fine-tune-your-own-bert-token-classification-model-06b1153fbf56):

    0: O => ordinary word
    1: B-DNA => beginning of a “DNA” term
    2: I-DNA => continuation of a “DNA” term
    3: B-RNA=> beginning of an “RNA” term
    4: I-RNA => contiunation of an “RNA” term
    5: B-protein => beginning of a “protein” term
    6: I-protein => continuation of a “protein” term
    7: B-cell_line => beginning of a “cell line” term
    8: I-cell_line => continuation of a “cell line” term
    9: B-cell_type => beginning of a “cell type” term
    10: I-cell_type => continuation of a “cell type” term

    We will lump B- and I- labels together - it'll be an easier task if
    the classifier doesn't have to figure out word position in the sentence.
"""

"\nDataset description: https://huggingface.co/datasets/jnlpba/jnlpba\n\nNLP for labeling biological terms.  Original labels (thanks to\nhttps://medium.com/@raj.pulapakura/fine-tune-your-own-bert-token-classification-model-06b1153fbf56):\n\n    0: O => ordinary word\n    1: B-DNA => beginning of a “DNA” term\n    2: I-DNA => continuation of a “DNA” term\n    3: B-RNA=> beginning of an “RNA” term\n    4: I-RNA => contiunation of an “RNA” term\n    5: B-protein => beginning of a “protein” term\n    6: I-protein => continuation of a “protein” term\n    7: B-cell_line => beginning of a “cell line” term\n    8: I-cell_line => continuation of a “cell line” term\n    9: B-cell_type => beginning of a “cell type” term\n    10: I-cell_type => continuation of a “cell type” term\n\n    We will lump B- and I- labels together - it'll be an easier task if\n    the classifier doesn't have to figure out word position in the sentence.\n"

In [23]:
!pip install transformers datasets evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6c2027874d40241d7c16671b2ae92a36473f6a947baf794775d395486ed16243
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:

from datasets import load_dataset

raw_dataset = load_dataset("siddharthtumre/jnlpba-split")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

validation.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Let's take a look at what a HuggingFace dataset looks like:

In [25]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 14690
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3856
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3856
    })
})

In [26]:
raw_dataset['train'][0]

{'id': '2427',
 'tokens': ['[',
  'An',
  'overexpression',
  'of',
  'retinoic',
  'acid',
  'receptor',
  'alpha',
  'blocks',
  'myeloid',
  'cell',
  'differentiation',
  'at',
  'the',
  'promyelocyte',
  'stage',
  ']'],
 'ner_tags': [0, 0, 0, 0, 9, 10, 10, 10, 0, 7, 8, 0, 0, 0, 0, 0, 0]}

In [27]:
# Some BERT code adapted from
# https://github.com/jalammar/jalammar.github.io/blob/master/notebooks/bert/A_Visual_Notebook_to_Using_BERT_for_the_First_Time.ipynb

import transformers as ppb

WEIGHTS = 'distilbert-base-uncased'
def get_tokenizer():
    return ppb.DistilBertTokenizer.from_pretrained(WEIGHTS)

tokenizer = get_tokenizer()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Note that the BERT tokenizer may break a word into parts if it doesn't recognize the whole word.  "Ohioization" becomes two tokens, "Ohio" and "##ization."

The tokenizer thinks of everything as a sentence and concatenates begin and end tokens to its tokenizations.  The following function strips these.

In [ ]:
def get_tokens(word, tokenizer):
    token_list = tokenizer.encode(word)
    return token_list[1:-1]

III.1, 9 points) Write a function dataset_to_bert_input_and_labels that takes one of the Huggingface datasets (the 'train' set), the tokenizer, and a maximum number of sentences, and returns a 2D array with as many sentences as there were in the data (or max_sentences, whichever is smaller) and as many columns as are necessary for the longest number of tokens, plus two.  Each token list should start with 101 (the CLS token) and end with 102 (end) - hence the +2.

In addition, force all the B- labels (odd numbers) to be the corresponding I- labels (even numbers one larger).

You can construct your first output as a list-of-lists at first, and in a second pass through the sentences, pad each of your lists with 0's, so that they are all the same length.  Then convert to 2D array.

So, for example, `dataset_to_bert_input_and_labels(raw_dataset['train'], tokenizer, 2)` should produce two return values - the first, a 2D array that looks like np.array([[101, ..., 0], [101, ..., 102]]), and the second, a list of label lists where the two lists are composed of 8's, 10's, and 0's.

In [ ]:

import numpy as np

def dataset_to_bert_input_and_labels(dataset, tokenizer, max_sentences):
    CLS_ID, SEP_ID = 101, 102

    token_rows, label_rows = [], []
    n = min(max_sentences, len(dataset))

    for idx in range(n):
        example = dataset[idx]
        words, ner_tags = example['tokens'], example['ner_tags']

        row_tokens = [CLS_ID]
        row_labels = [0]  

        for word, tag in zip(words, ner_tags):
            if tag % 2 == 1:
                tag = tag + 1

            sub_tokens = tokenizer.encode(word)[1:-1]  
            if len(sub_tokens) == 0:
                continue

            row_tokens.extend(sub_tokens)
            row_labels.extend([tag] * len(sub_tokens))

        row_tokens.append(SEP_ID)
        row_labels.append(0) 
        token_rows.append(row_tokens)
        label_rows.append(row_labels)

    max_len = max(len(row) for row in token_rows)

    padded_tokens = np.zeros((n, max_len), dtype=np.int64)
    padded_labels = []

    for i, (tok_row, lab_row) in enumerate(zip(token_rows, label_rows)):
        length = len(tok_row)
        padded_tokens[i, :length] = tok_row
        padded_labels.append(lab_row + [0] * (max_len - length))

    return padded_tokens, padded_labels

In [ ]:
dataset_to_bert_input_and_labels(raw_dataset['train'], tokenizer, 2) 

(array([[  101,  1031,  2019,  2058, 10288, 20110,  3258,  1997,  2128,
         25690,  2594,  5648, 10769,  6541,  5991,  2026, 18349,  3593,
          3526, 20582,  2012,  1996, 20877,  6672,  4135,  5666,  2618,
          2754,  1033,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [  101,  2057,  2031,  3491,  3041,  2008,  1050,  2546,  1011,
          1049,  1010,  1996,  7975, 24004, 21197,  1997,  1039,  1013,
          1041,  2497,  2361,  8247,  1010,  2003,  4919,  5228,  1999,
          2026, 18349,  8202, 10085, 21252,  1998,  1041, 20049,  3630,
         21850, 10415,  4442,  1997,  1996, 19610, 10610,  6873,  2666,
          4588,  2291,  1012,   102]]),
 [[0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   10,
   10,
   10,
   10,
   10,
   10,
   0,
   8,
   8,
   8,
   8,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0

The following code should then produce a $800 \times 180 \times 768$ tensor - 800 sentences with at most 180 tokens, each of which has a 768 element vector associated with it.  This may take a little while as each token list is run through the pretrained BERT network.

In [ ]:
import torch

def get_model():
    return ppb.DistilBertModel.from_pretrained(WEIGHTS)

def get_bert_vectors(model, padded_tokens):
    mask = torch.tensor(np.where(padded_tokens != 0, 1, 0))
    with torch.no_grad():
        word_vecs = model(torch.tensor(padded_tokens).to(torch.int64), attention_mask=mask)
    return word_vecs[0][:,:,:].numpy()

train_input, labels = dataset_to_bert_input_and_labels(raw_dataset['train'], tokenizer, 800)
model = get_model()
bert_result = get_bert_vectors(model, train_input)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
print(bert_result.shape) # Expect (800, 180, 768)

(800, 180, 768)


III.2, 3 points) In your own words, what is the use of the vector associated with the 0th token, the CLS token?




**TODO**

the vector associated with the oth token or the CLS token is used to give the meaning or a summary of the entire sentence as a whole in order to be able to classify it, which it can then make predictions as the sentence as a whole.

III.3, 8 points) Now you'll write the glue that connects the BERT part of the pipeline to some off-the-shelf ML.  Write `labels_and_bert_to_sklearn(labels, bert_result)` which takes the label list-of-lists you produced earlier and the tensor that was the result of the get_bert_vectors() call, and produce a single $W \times 768$ features matrix and a $W$-element labels array, such that both could be used as features and labels in scikit-learn.  (By $W$, we mean the total number of words in the data.)

In [ ]:

import numpy as np

def labels_and_bert_to_sklearn(labels, bert_result):
    feature_list, label_list = [], []

    for i in range(bert_result.shape[0]):
        norms = np.linalg.norm(bert_result[i], axis=1)
        non_pad = np.where(norms > 1e-6)[0]

        if len(non_pad) == 0:
            continue
        word_positions = non_pad[1:-1]
        for j in word_positions:
            feature_list.append(bert_result[i, j, :])
            label_list.append(labels[i][j])

    return np.array(feature_list), np.array(label_list)


In [34]:
bert_features_train, bert_labels_train = labels_and_bert_to_sklearn(labels, bert_result)

III.4, 4 points) Call the following code block to get test data as well.  Then train a scikit-learn RandomForestClassifier with 200 estimators and random state 340 - this will take about 6 minutes on Colab - and evaluate the classifier on the test set.  You can expect an accuracy of about 80%.

In [35]:
test_input, test_labels = dataset_to_bert_input_and_labels(raw_dataset['validation'], tokenizer, 100)
bert_result_test = get_bert_vectors(model, test_input)
bert_features_test, bert_labels_test = labels_and_bert_to_sklearn(test_labels, bert_result_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bert_rf = RandomForestClassifier(n_estimators=200, random_state=340)
bert_rf.fit(bert_features_train, bert_labels_train)

print(f"BERT + RF accuracy: {bert_rf.score(bert_features_test, bert_labels_test):.4f}")


BERT + RF accuracy: 0.9318


# IV.  Off-the-shelf fine-tuned model (6 points)

If you want to fine-tune a BERT model, you can follow a web tutorial [here](https://learnopencv.com/fine-tuning-bert/#aioseo-fine-tuning-bert-on-the-arxiv-abstract-classification-dataset), but this takes a while.  Let's just see how much better we can do if we have a fine-tuned BERT model, versus the word2vec approach we started with.  For common datasets like CONLL2003, it's possible to find models others have already trained on the HuggingFace website.

1, 6 pts) Make a prediction to yourself about what kind of F1 scores you might expect to see from this large fine-tuned transformer model.  Then run the two code boxes below to load a fine-tuned CONLL2003 NER model from HuggingFace ("dbmdz/bert-large-cased-finetuned-conll03-english")and evaluate it on the CONLL2003 test data.  For each class (besides O = ordinary), compare the f1 score to the f1 score for the same class in the word2vec classifier of part I.

* Roughly how much of a bump in f1 score do we see for each classification, and on average?

* Is the model's final performance better or worse than you expected, and why?

In [37]:
# More on fine-tuning token-classification models at https://medium.com/@raj.pulapakura/fine-tune-your-own-bert-token-classification-model-06b1153fbf56

from transformers import pipeline

token_classifier = pipeline(
  "token-classification",
  "dbmdz/bert-large-cased-finetuned-conll03-english",
  aggregation_strategy="simple",
)

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [38]:
# https://huggingface.co/docs/evaluate/en/base_evaluator

from datasets import load_dataset
from evaluate import evaluator
from transformers import AutoModelForSequenceClassification, pipeline

data = load_dataset("eriktks/conll2003", split="test", revision="refs/convert/parquet").shuffle(seed=340).select(range(1000))
task_evaluator = evaluator("token-classification")

eval_results = task_evaluator.compute(
    model_or_pipeline="dbmdz/bert-large-cased-finetuned-conll03-english",
    data=data,
    metric="seqeval"
)

print(eval_results)

conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'LOC': {'precision': np.float64(0.9370229007633588), 'recall': np.float64(0.9334600760456274), 'f1': np.float64(0.9352380952380954), 'number': np.int64(526)}, 'MISC': {'precision': np.float64(0.8102564102564103), 'recall': np.float64(0.797979797979798), 'f1': np.float64(0.8040712468193385), 'number': np.int64(198)}, 'ORG': {'precision': np.float64(0.8776824034334764), 'recall': np.float64(0.9149888143176734), 'f1': np.float64(0.895947426067908), 'number': np.int64(447)}, 'PER': {'precision': np.float64(0.9691629955947136), 'recall': np.float64(0.9606986899563319), 'f1': np.float64(0.9649122807017544), 'number': np.int64(458)}, 'overall_precision': np.float64(0.9139719341061623), 'overall_recall': np.float64(0.9195825659914058), 'overall_f1': np.float64(0.9167686658506732), 'overall_accuracy': 0.8713410991636799, 'total_time_in_seconds': 138.16226240800097, 'samples_per_second': 7.23786642293786, 'latency_in_seconds': 0.13816226240800097}


* **TODO F1 differences**

On average the fine-tuned model gives roughly a +0.08 bump in F1 for all classes
the differences are as follos:
PER went from 0.9018 to 0.9649 (+0.06)
ORG went from 0.7692 to 0.8959 (+0.13)
LOC went from 0.8675 to 0.9352 (+0.07)
MISC went from 0.7442 to 0.8041 (+0.06)

* **TODO compare to your predictions**
in general the fine tuned performed better, org improved the most comapred to the rest but overall they all improved or got bumped a bit.
ORG improved the most, which is explained because organization names are the most context-dependent. meanwhile  MISC still had
the lowest F1 (0.80) even with fine-tuning, confirming that miscellaneous
entities are genuinely the hardest class no matter the approach.


# AI Statement (2 pts)

Please briefly describe whether and how you used generative AI for this assignment.  You will not be penalized for your answer - this is mostly so the course can adapt to AI use.

**TODO**

i did use generative AI. i specifically used claude. i asked  a bunch of questions regarding syntax and the structure of natural language processi as well as some questions about uploading document to googel colab. i used it to explain to me the errors i got and sometimes it would help complete some of my gaps for example whe i would be missing import statments